# GEE Pipeline — Clean + Rename + Merge

Run cells **1 → 2 → 3** in order.

**Output:** `Merged_Lake_Water_Quality_Master.csv` — **353 rows, 39 lakes**

In [30]:
# ═══════════════════════════════════════════
# CELL 1 — CLEAN GEE CSV
# ═══════════════════════════════════════════
import pandas as pd, re

GEE_CSV    = 'GEE-Dataset.csv'
MASTER_CSV = 'master_dataset.csv'
OUT_CLEAN  = 'GEE-Dataset-Cleaned.csv'
OUT_FINAL  = 'GEE-DaFinataset-Cleaned_l.csv'
OUT_MERGED = 'Merged_Lake_Water_Quality_Master.csv'

df = pd.read_csv(GEE_CSV)
print(f'Loaded              : {df.shape}')

# Auto-detect value columns (everything except ID/flag columns)
ID_COLS  = ['system:index','LAKE_NAME','MONTH_YEAR','YEAR','MONTH','n_s2_images','data_flag']
VAL_COLS = [c for c in df.columns if c not in ID_COLS]

# Drop rows where ALL value columns are NaN (no S2 data that month)
df = df.dropna(subset=VAL_COLS, how='all')
print(f'After drop all-NaN  : {df.shape}')

# Drop rows with zero valid water pixels
count_col = next((c for c in df.columns if c.endswith('_count')), None)
if count_col:
    df = df[df[count_col] > 0]
    print(f'After drop count==0 : {df.shape}  [{count_col}]')

# Drop admin-only columns
df = df.drop(columns=[c for c in ['system:index','data_flag'] if c in df.columns])

df.to_csv(OUT_CLEAN, index=False)
print(f'Saved → {OUT_CLEAN}')
df.head()

Loaded              : (9280, 20)
After drop all-NaN  : (9280, 20)
Saved → GEE-Dataset-Cleaned.csv


,name,stn,scene_date,lat,lon,B2,B3,B4,B5,B6,B7,B8,B11,B12,NDWI,MNDWI,NDCI,NDTI,Red_Green,RedEdge_Ratio
0,Devasandra Lake,DVS,2023-08-16,13.0220,77.6553,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Hebbal Lake,HBL,2023-08-16,13.0455,77.5961,0.241170,0.255092,0.261019,0.294573,0.347088,0.376278,0.364230,0.323677,0.247185,-0.176924,-0.119681,0.060891,0.012280,1.025438,1.131280
2,Jakkur Lake,JKR,2023-08-16,13.0648,77.6004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Nagavara Lake,NGV,2023-08-16,13.0318,77.6130,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Chelekere Lake,CLK,2023-08-16,13.0396,77.6109,0.151643,0.167493,0.172714,0.207338,0.277629,0.317454,0.311684,0.324453,0.291991,-0.300655,-0.331611,0.111064,0.009771,1.024984,1.296459


In [31]:
# ═══════════════════════════════════════════
# CELL 2 — RENAME LAKE NAMES
# ═══════════════════════════════════════════
import pandas as pd

df_gee = pd.read_csv(OUT_CLEAN)

rename_map = {
    'Mahadevepura'             : 'Mahadevpura',
    'Agara'                    : 'Agaram',
    'Allasandra'               : 'Allalasandra',
    'Bhoganahalli'             : 'Bhoganhalli',
    'Chellakere'               : 'Chelekere',
    'Devarabisanahalli'        : 'Devarabeesanahalli',
    'Gangashetti'              : 'Gangashetty',
    'Nagawara'                 : 'Nagavara',
    'Panattur'                 : 'Panathur',
    'Parapana'                 : 'Parappana Agrahara',
    'Sarraki'                  : 'Sarakki',
    'Sowl'                     : 'Soulkere',
    'Vibhutipura'              : 'Vibhuthipura',
    'Vijinapura'               : 'Vijanapura',
    'Arekere'                  : 'Arakere',
    'Thubarahalli'             : 'Tubarahalli',
    'Gangashetti / Devasandra' : 'Devasandra',
    'Kembathahali'             : 'Kembhatahali',
    'Nallurahallikere'         : 'Nallurahalli',
    'Doraikere'                : 'Uttarahalli Doraikere',
    # add more here as needed
}

df_gee['name'] = (
    df_gee['name']
    .map(lambda x: rename_map.get(str(x).strip(), str(x).strip()))
    .str.title().str.strip()
)

df_gee.to_csv(OUT_FINAL, index=False)
print(f'Rows: {len(df_gee)}  Unique lakes: {df_gee["name"].nunique()}')
print(f'Saved → {OUT_FINAL}')

Rows: 9280  Unique lakes: 80
Saved → GEE-DaFinataset-Cleaned_l.csv


In [33]:
# ═══════════════════════════════════════════
# CELL 3 — MERGE WITH MASTER DATASET (FIXED)
# ═══════════════════════════════════════════
import pandas as pd, re

df_gee    = pd.read_csv(OUT_FINAL)
df_master = pd.read_csv(MASTER_CSV)

# ── FIX 1: Rename 'name' to 'LAKE_NAME' in GEE so suffixes work properly ─────
if 'name' in df_gee.columns:
    df_gee = df_gee.rename(columns={'name': 'LAKE_NAME'})

print(f'GEE rows    : {len(df_gee)}')
print(f'Master rows : {len(df_master)}')

# ── Normalise lake names for matching ────────────────────────────────────────
def norm_name(s):
    if pd.isna(s): return ''
    s = str(s).lower()
    s = re.sub(r'\(un-named lake on map\)', '', s)
    s = re.sub(r'\b(lake|kere|tank)\b', '', s)
    s = re.sub(r'[-_/]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

# Rename master lake column
df_master = df_master.rename(columns={'NAMEOFMONITORINGLOCATION': 'LAKE_NAME'})

# Create matching keys for names
df_gee['_key_name']    = df_gee['LAKE_NAME'].map(norm_name)
df_master['_key_name'] = df_master['LAKE_NAME'].map(norm_name)

# ── FIX 2: Standardize Date Formats to MM/YYYY ───────────────────────────────
# Convert GEE 'YYYY-MM-DD' -> 'MM/YYYY'
df_gee['_key_month'] = pd.to_datetime(df_gee['scene_date']).dt.strftime('%m/%Y')

# Convert Master 'MM/YY' -> 'MM/YYYY' 
# (format='%m/%y' handles the 2-digit year correctly)
df_master['_key_month'] = pd.to_datetime(df_master['MONTH_YEAR'], format='%m/%y', errors='coerce').dt.strftime('%m/%Y')

# Fallback for any Master dates already in MM/YYYY format
df_master['_key_month'] = df_master['_key_month'].fillna(
    pd.to_datetime(df_master['MONTH_YEAR'], format='%m/%Y', errors='coerce').dt.strftime('%m/%Y')
)
# ─────────────────────────────────────────────────────────────────────────────

# ── MERGE ────────────────────────────────────────────────────────────────────
merged_df = pd.merge(
    df_gee, df_master,
    on=['_key_name', '_key_month'],
    how='inner',
    suffixes=('_GEE', '_master')
)

# Drop helper key columns
merged_df = merged_df.drop(columns=['_key_name','_key_month'], errors='ignore')

print(f'\nMerged rows   : {len(merged_df)}')
print(f'Merged cols   : {merged_df.shape[1]}')

if merged_df.empty:
    print('\n⚠ No rows matched. Check if Lake names or Dates exist in both files.')
    print('Sample GEE Keys:', df_gee['_key_month'].unique()[:5])
    print('Sample Master Keys:', df_master['_key_month'].unique()[:5])
else:
    # Now LAKE_NAME_GEE will exist because LAKE_NAME was present in both before merge
    print(f'Matched lakes : {merged_df["LAKE_NAME_GEE"].nunique()}')
    
    show_cols = [c for c in ['LAKE_NAME_GEE','scene_date','MONTH_YEAR_master','DO','BOD'] if c in merged_df.columns]
    print("\n--- Sample Merged Data ---")
    print(merged_df[show_cols].head(10).to_string())

merged_df.to_csv(OUT_MERGED, index=False)
print(f'\nSaved → {OUT_MERGED}')

GEE rows    : 9280
Master rows : 2924

Merged rows   : 5705
Merged cols   : 54
Matched lakes : 70

--- Sample Merged Data ---
          LAKE_NAME_GEE  scene_date   DO   BOD
0       Devasandra Lake  2023-08-16  5.0   4.0
1           Hebbal Lake  2023-08-16  4.2  11.0
2           Jakkur Lake  2023-08-16  2.8  23.0
3         Nagavara Lake  2023-08-16  6.1   4.0
4        Chelekere Lake  2023-08-16  6.6   4.0
5        Bellandur Lake  2023-08-16  0.3  18.0
6          Varthur Lake  2023-08-16  0.3  52.0
7         Madiwala Lake  2023-08-16  4.8   8.0
8  Kaikondanahalli Lake  2023-08-16  4.4   9.0
9     Puttenahalli Lake  2023-08-16  4.2   4.0

Saved → Merged_Lake_Water_Quality_Master.csv
